In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import numpy as np
import seaborn as sns
from functions import abbr_substitutor, make_sentence_df, count_words, split_into_sentences
import collections
import re
from collections import Counter
import statistics
from nltk.tokenize import sent_tokenize


pd.set_option('display.max_colwidth', None)
#replace with path to the cleaned careplan reports
df = pd.read_csv('careplan_reports_cleaned.csv', sep=',', encoding='latin-1')
df

In [ ]:
#Preprocess the messy data
df['comment'] = df['comment'].fillna('')
df = df.drop(df[df['comment'] == ''].index)
df['preprocessed'] = df['comment'].apply(abbr_substitutor)
preprocessed_df = df[['comment', 'preprocessed']]
preprocessed_df = preprocessed_df.drop(preprocessed_df[preprocessed_df['comment'] == ''].index)
preprocessed_df

In [ ]:
#preprocessed_df=pd.read_csv('E:\Data_exploration\careplan_reports_preprocessed.csv', encoding='latin-1')

Data distribution

In [ ]:
preprocessed_df['num_words'] = preprocessed_df['comment'].apply(lambda x: len(str(x).split()))
pruned_df = preprocessed_df[(preprocessed_df['num_words'] < 150) & (preprocessed_df['num_words'] > 2)]#['comments']
pruned_df.to_csv('careplan_reports_preprocessed.csv', index=False)

In [ ]:
def plot_distribution(df, max_x=50):
    # Extract the column of interest
    data = df['num_words']
    # Count the occurrences of each unique value
    value_counts = data.value_counts()

    # Plot the bar chart
    plt.bar(value_counts.index, value_counts.values, color='g', alpha=0.6)
    plt.xlabel('Words in text')
    plt.ylabel('Frequency')
    plt.title('Frequency of entry length')
    plt.grid(True)

    # Set x-axis limits to focus on the majority of the points between 0 and 100
    plt.xlim(0, max_x)

    plt.show()

In [ ]:
def calculate_word_statistics(df, column_name):
    # Calculate the mean and standard deviation of the number of words per entry
    mean_words = df['num_words'].mean()
    std_words = df['num_words'].std()
    median_words = df['num_words'].median()
    min_words = df['num_words'].min()
    max_words = df['num_words'].max()
    percentile_25 = df['num_words'].quantile(0.25)
    percentile_75 = df['num_words'].quantile(0.75)
    percentile_5 = df['num_words'].quantile(0.05)
    percentile_95 = df['num_words'].quantile(0.95)

    print("Average number of words per entry:", mean_words)
    print("Standard deviation of words per entry:", std_words)
    print("Median number of words per entry:", median_words)
    print("Minimum number of words per entry:", min_words)
    print("Maximum number of words per entry:", max_words)
    print("25th percentile of words per entry:", percentile_25)
    print("75th percentile of words per entry:", percentile_75)
    print("5th percentile of words per entry:", percentile_5)
    print("95th percentile of words per entry:", percentile_95)

    # Return the calculated statistics
    return mean_words, std_words, median_words, min_words, max_words, percentile_25, percentile_75, percentile_5, percentile_95

# Assuming you have a DataFrame called 'preprocessed_df'
print('preprocessed_df')
mean_words, std_words, median_words, min_words, max_words, percentile_25, percentile_75, percentile_5, percentile_95 = calculate_word_statistics(pruned_df, 'comments')

In [ ]:
#distribution of text lengths.
print('distribution of entire comment length')
plot_distribution(pruned_df, 150)

Split data into sentences

In [ ]:
#Statistics on the sentence dataset
pruned_df_sentences = make_sentence_df(pruned_df, 'comment')
pruned_df_sentences = pruned_df_sentences[~pruned_df_sentences['num_words'].isna()]
print('preprocessed_sentences')
mean_words, std_words, median_words, min_words, max_words, percentile_25, percentile_75, percentile_5, percentile_95 = calculate_word_statistics(pruned_df_sentences, 'sentences')
plot_distribution(pruned_df_sentences)

#save_df = pruned_df_exploded[['num_words', 'sentences']]
#can be used for clustering
#save_df.to_csv('careplan_sentences_preprocessed.csv', index=False)

In [ ]:
#Compare with example generated data
example_gen_df = pd.read_pickle("E:\Data_exploration\GitHub\paraphraser_code\data\generated\generated_correct_medical_augmented_5k_v1.pkl")
example_df = example_gen_df.sample(frac=1).reset_index(drop=True)
sentences_examples_df = make_sentence_df(example_df, 'Generated')
sentences_examples_df = sentences_examples_df[['sentences', 'num_words']]
#sentences_examples_df.to_csv('sentences_examples.csv')
sentences_examples_df.head(1)
plot_distribution(sentences_examples_df)

In [ ]:
#compare with keyword generated text
keyword_generated_df = pd.read_csv('E:\Data_exploration\GitHub\paraphraser_code\data\generated\generated_keyword3_v4.csv')
keyword_generated_df_exploded = make_sentence_df(keyword_generated_df, 'Generated')
mean_words, std_words, median_words, min_words, max_words, percentile_25, percentile_75, percentile_5, percentile_95 = calculate_word_statistics(keyword_generated_df_exploded, 'sentences')
plot_distribution(keyword_generated_df_exploded)

Capture vocabulary of real data for spellchecker

In [ ]:
#Count words in the corpus
def get_sorted_word_counts(df, column_name):
    word_counts = collections.defaultdict(int)
    for text in df[column_name]:
        if isinstance(text, str):
            words = re.findall(r'\b\w+\b', text)
            for word in words:
                word_counts[word.lower()] += 1
    sorted_word_counts = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
    sorted_words = {word: count for word, count in sorted_word_counts}
    return sorted_words

sorted_words = get_sorted_word_counts(pruned_df,'comments')
sorted_words

In [ ]:
def filter_words_by_occurrences(word_counts, min_occurrences):
    filtered_words = {word: count for word, count in word_counts.items() if count >= min_occurrences}
    return filtered_words

In [ ]:
#Filtered words has all words up to a certain occurance threshold
total_counts = sum(sorted_words.values())
total_counts

#only words that occur every 2.5 million words.
min_occurrences_threshold = total_counts/2500000
filtered_words = filter_words_by_occurrences(sorted_words, min_occurrences_threshold)
filtered_words

In [ ]:
def read_counter_from_file(file_path):
    # Read the content of the text file and create a Counter dictionary
    counter = Counter()
    with open(file_path, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) == 2:  # Check if the line has the expected format
                word, freq = parts
                counter[word] = int(freq)
    return counter

def combine_counters(base_counter, custom_counter):
    base_counter.update(custom_counter)
    return base_counter

def save_counter_to_file(counter, file_path):
    with open(file_path, 'w') as file:
        for word, freq in counter.items():
            file.write(f"{word} {freq}\n")
#found at opentaal github
freq_path = "E:/Data_exploration/GitHub/paraphraser_code/data_preparation/spellchecker/nl.txt"
counter_from_txt = read_counter_from_file(freq_path)

new_counter = combine_counters(counter_from_txt, filtered_words)

#saved you get the nl_with_corpus.txt file with words and occurances.
#save_counter_to_file(new_counter, 'nl_with_corpus.txt')

Capture distribution of sentence lengths original data --- SCRAPPED IDEA

In [ ]:
counts = preprocessed_df['num_words'].value_counts()
counts_df = counts.reset_index()

# Rename the columns to give them specific names
counts_df.columns = ['number', 'number_occurance']
#counts_df.to_csv('pruned_distribution3to110.csv', index=False)

In [ ]:
pd.read_csv('pruned_distribution3to110.csv')